In [1]:
import pandas as pd
import numpy as np
import math

# 1. 模拟生成高价值航电备件数据 (High-Value Avionics Data)
def generate_mro_data(n_parts=50):
    np.random.seed(42)
    data = []
    for i in range(n_parts):
        data.append({
            'Part_ID': f'HW-AV-{1000+i}', # 模拟 Honeywell (HW) 航电零件编号
            'Unit_Cost_USD': round(np.random.uniform(5000, 50000), 2), # 单价极高
            'Avg_Monthly_Demand': round(np.random.uniform(5, 50)),     # 月均需求
            'Demand_Std_Dev': round(np.random.uniform(2, 15)),         # 需求波动
            'Lead_Time_Months': round(np.random.uniform(1, 4), 1)      # 供应商交期
        })
    return pd.DataFrame(data)

# 2. 核心算法：精益库存优化 (Lean Inventory Optimization)
def optimize_inventory(df, target_service_level=0.95):
    # 95% 服务水平对应的 Z-Score 约为 1.645 (标准正态分布)
    z_score = 1.645

    # 计算动态安全库存 (Safety Stock)
    # 公式: Z * sqrt(Lead_Time) * Standard_Deviation
    df['Optimized_Safety_Stock'] = np.ceil(
        z_score * np.sqrt(df['Lead_Time_Months']) * df['Demand_Std_Dev']
    )

    # 计算重新订货点 (Reorder Point - ROP)
    df['Reorder_Point'] = np.ceil(
        (df['Avg_Monthly_Demand'] * df['Lead_Time_Months']) + df['Optimized_Safety_Stock']
    )

    # 模拟传统“拍脑袋”库存法 vs 数据驱动优化法的资金占用差异
    # 假设传统方法为了安全，一律压 3 个月的平均需求量作为安全库存
    df['Legacy_Safety_Stock'] = np.ceil(df['Avg_Monthly_Demand'] * 3)

    df['Legacy_Capital_Tied_USD'] = df['Legacy_Safety_Stock'] * df['Unit_Cost_USD']
    df['Optimized_Capital_Tied_USD'] = df['Optimized_Safety_Stock'] * df['Unit_Cost_USD']

    return df

# 3. 执行分析与资金释放报告
mro_df = generate_mro_data()
optimized_df = optimize_inventory(mro_df)

# 计算宏观财务指标
total_legacy_capital = optimized_df['Legacy_Capital_Tied_USD'].sum()
total_optimized_capital = optimized_df['Optimized_Capital_Tied_USD'].sum()
capital_saved = total_legacy_capital - total_optimized_capital
reduction_pct = (capital_saved / total_legacy_capital) * 100

print("--- MRO Spare Parts Optimization Report ---")
print(f"Total Parts Analyzed: {len(optimized_df)}")
print(f"Legacy Capital Tied Up: ${total_legacy_capital:,.2f}")
print(f"Optimized Capital Tied Up: ${total_optimized_capital:,.2f}")
print(f"🚀 Capital Freed Up (Cost Avoidance): ${capital_saved:,.2f} ({reduction_pct:.2f}% Reduction)\n")

print("--- Top 5 Parts with Highest Capital Reduction ---")
optimized_df['Capital_Savings'] = optimized_df['Legacy_Capital_Tied_USD'] - optimized_df['Optimized_Capital_Tied_USD']
top_savings = optimized_df.sort_values(by='Capital_Savings', ascending=False).head(5)
print(top_savings[['Part_ID', 'Unit_Cost_USD', 'Legacy_Safety_Stock', 'Optimized_Safety_Stock', 'Capital_Savings']])

--- MRO Spare Parts Optimization Report ---
Total Parts Analyzed: 50
Legacy Capital Tied Up: $98,287,510.86
Optimized Capital Tied Up: $27,855,893.99
🚀 Capital Freed Up (Cost Avoidance): $70,431,616.87 (71.66% Reduction)

--- Top 5 Parts with Highest Capital Reduction ---
       Part_ID  Unit_Cost_USD  Legacy_Safety_Stock  Optimized_Safety_Stock  \
30  HW-AV-1030       41334.81                135.0                    12.0   
13  HW-AV-1013       47277.45                135.0                    33.0   
28  HW-AV-1028       46836.39                123.0                    32.0   
48  HW-AV-1048       45518.81                 99.0                    14.0   
20  HW-AV-1020       43839.65                 99.0                    11.0   

    Capital_Savings  
30       5084181.63  
13       4822299.90  
28       4262111.49  
48       3869098.85  
20       3857889.20  


### ** Simulation Results**
Based on a recent simulation of 50 high-value avionics SKUs using statistical demand modeling:
* **Legacy Capital Tied Up**: **$98.28 Million** (Using legacy static 3-month buffering)
* **Optimized Capital Tied Up**: **$27.85 Million** (Using Z-score dynamic safety stock)
* ** Capital Freed Up (Cost Avoidance)**: **$70.43 Million (71.66% Reduction)**
* **Impact**: Successfully demonstrated how precision statistical forecasting can maintain a 95% service level to prevent AOG (Aircraft on Ground) while releasing massive amounts of working capital for operational reinvestment.